# Moroccan Legal RAG -- Benchmark Suite

> Comprehensive evaluation of the Moroccan Legal RAG system using five core metrics:
> **Context Precision**, **Context Recall**, **Answer Relevance**, **Faithfulness**, and **Answer Correctness**.

## Metrics Overview

| Metric | What it measures | Formula |
|--------|-----------------|---------|
| **Context Precision** | Relevance of retrieved docs | `|relevant cap retrieved| / |retrieved|` |
| **Context Recall** | Coverage of all relevant docs | `|relevant cap retrieved| / |relevant|` |
| **Answer Relevance** | LLM alignment with query & context | Cosine similarity (Q, A) via embeddings |
| **Faithfulness** | Grounding of claims in retrieved docs | `|supported claims| / |total claims|` |
| **Answer Correctness** | Match with ground truth | BERTScore F1 + ROUGE-L |

In [21]:
# =============================================================================
# Section 1: Import Required Libraries
# =============================================================================

from __future__ import annotations

import json
import sys
import os
import time
import re
import textwrap
from pathlib import Path
from typing import List, Dict, Optional, Set, Tuple
from dataclasses import dataclass, field
from collections import defaultdict
import warnings
warnings.filterwarnings("ignore")

# Data & numerics
import numpy as np
import pandas as pd

# Visualization
import matplotlib
matplotlib.use("Agg")  # non-interactive backend for headless envs
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from matplotlib.patches import FancyBboxPatch

# NLP & evaluation
# Note: rouge-score and bert-score are optional; install with:
#   pip install rouge-score bert-score sentence-transformers
try:
    from rouge_score import rouge_scorer
    _HAS_ROUGE = True
except ImportError:
    _HAS_ROUGE = False
    print("[WARN] rouge-score not installed. ROUGE-L will be skipped.")

try:
    from bert_score import score as bert_score_fn
    _HAS_BERT_SCORE = True
except ImportError:
    _HAS_BERT_SCORE = False
    print("[WARN] bert-score not installed. BERTScore will be skipped.")

# Project imports -- resolve repo root and fix CWD
# The notebook lives in benchmark/; we need the repo root for all relative paths
REPO_ROOT = Path(os.getcwd()).resolve()
if REPO_ROOT.name == "benchmark" or "benchmark" in str(REPO_ROOT):
    while REPO_ROOT.name != "Analyse-des-lois-justice-Maroc":
        REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
os.chdir(str(REPO_ROOT))  # critical: src/utils/config.py uses relative Path("config")

from dotenv import load_dotenv
load_dotenv(REPO_ROOT / ".env")

# DB & retrieval
from src.db.session import get_connection
from src.core.embedding import create_embedder
from src.retrieval.hybrid_search import search, create_retrieval_strategy
from src.generation.factory import create_llm

# Configure matplotlib style
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 150,
    "savefig.bbox": "tight",
    "font.family": "DejaVu Sans",
})

# Output directory for figures & results
OUTPUT_DIR = REPO_ROOT / "benchmark" / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"REPO_ROOT = {REPO_ROOT}")
print(f"Imports ready. Output -> {OUTPUT_DIR}")

REPO_ROOT = /home/regisx001/Analyse-des-lois-justice-Maroc
Imports ready. Output -> /home/regisx001/Analyse-des-lois-justice-Maroc/benchmark/output


## Section 2: Load Benchmark Dataset

Load the benchmark JSON and flatten it into a DataFrame for evaluation.

In [22]:
# =============================================================================
# Section 2: Load Benchmark Dataset
# =============================================================================

BENCHMARK_FILE = REPO_ROOT / "benchmark" / "retrieval_benchmark.json"

with open(BENCHMARK_FILE, "r", encoding="utf-8") as f:
    benchmark = json.load(f)

meta = benchmark["meta"]
questions = benchmark["questions"]

print(f"Benchmark loaded: {meta['description']}")
print(f"   Version: {meta['version']}  |  Questions: {meta['total_questions']}")
print(f"   Domains: {', '.join(meta['domains'])}")
print(f"   Difficulty: {', '.join(meta['difficulty_levels'])}")

# Flatten to DataFrame
rows = []
for q in questions:
    for art in q["expected_articles"]:
        rows.append({
            "id": q["id"],
            "domain": q["domain"],
            "difficulty": q["difficulty"],
            "tags": q["tags"],
            "question_ar": q["question_ar"],
            "expected_answer": q["expected_answer"],
            "expected_article_number": art["article_number"],
            "expected_law_name": art["law_name"],
            "query_variations": q.get("query_variations", []),
        })

df_bench = pd.DataFrame(rows)

# Aggregate expected articles per question
per_question = df_bench.groupby("id").agg(
    domain=("domain", "first"),
    difficulty=("difficulty", "first"),
    question_ar=("question_ar", "first"),
    expected_answer=("expected_answer", "first"),
    expected_articles=("expected_article_number", lambda x: list(x)),
    expected_laws=("expected_law_name", lambda x: list(x)),
    query_variations=("query_variations", "first"),
).reset_index()

print(f"\nPer-question dataset: {len(per_question)} unique questions")
print(f"   Total expected article references: {len(rows)}")
per_question.head(3)

Benchmark loaded: Enhanced benchmark dataset for Moroccan Legal RAG retrieval system — v2.0 with multi-article queries
   Version: 2.0  |  Questions: 32
   Domains: criminal, civil, commercial, traffic, judicial, tax
   Difficulty: easy, medium, hard

Per-question dataset: 32 unique questions
   Total expected article references: 52


,id,domain,difficulty,question_ar,expected_answer,expected_articles,expected_laws,query_variations
0,CIV-001,civil,easy,هل يمكن للمحكمة دعوة الأطراف إلى الوساطة لحل ا...,نعم، يمكن للمحكمة أن تدعو الأطراف إلى حل النزا...,[2],[code civil marocain ar],"[الوساطة في القانون المغربي, دعوة المحكمة للوس..."
1,CIV-002,civil,medium,ما هي قواعد الاختصاص القيمي للمحاكم الابتدائية...,تحكم المحكمة الابتدائية ابتدائيا وانتهائيا حتى...,[28],[code civil marocain ar],"[الاختصاص القيمي المحاكم الابتدائية, قيمة الدع..."
2,CIV-003,civil,hard,في أي الحالات تكون النيابة العامة طرفا أصليا ف...,تكون النيابة العامة طرفا أصليا في القضايا المت...,[14],[code civil marocain ar],"[النيابة العامة طرف أصلي, حالات تدخل النيابة ا..."


## Section 3: Define Context Precision / Relevance Metric

**Context Precision** measures whether the retrieved documents are actually relevant to the query.
We compute this as:

$$\text{Precision@k} = \frac{|\text{relevant articles} \cap \text{retrieved articles}|}{|\text{retrieved articles}|}$$

We also compute **MRR** (Mean Reciprocal Rank) and **NDCG@k** for ranking quality.

In [23]:
# =============================================================================
# Section 3: Context Precision / Relevance Metric
# =============================================================================

def compute_context_precision(
    retrieved_articles: List[str],
    expected_articles: List[str],
    k: int = 5,
) -> Dict[str, float]:
    """Compute Precision@k, MRR, and NDCG for retrieved articles.

    Args:
        retrieved_articles: List of article numbers retrieved (ordered by rank).
        expected_articles: List of expected (ground-truth) article numbers.
        k: Cutoff rank.

    Returns:
        Dict with precision@k, mrr@k, ndcg@k, hit@k.
    """
    expected_set = set(expected_articles)
    if not expected_set:
        return {"precision@k": 0.0, "mrr@k": 0.0, "ndcg@k": 0.0, "hit@k": 0.0}

    top_k = retrieved_articles[:k]

    # Precision@k
    relevant_count = sum(1 for art in top_k if art in expected_set)
    precision_at_k = relevant_count / min(k, len(top_k)) if top_k else 0.0

    # MRR@k
    mrr = 0.0
    for rank, art in enumerate(top_k, start=1):
        if art in expected_set:
            mrr = 1.0 / rank
            break

    # NDCG@k -- binary relevance (1 if in expected set, else 0)
    dcg = 0.0
    idcg = 0.0
    for rank, art in enumerate(top_k, start=1):
        rel = 1.0 if art in expected_set else 0.0
        dcg += (2**rel - 1) / np.log2(rank + 1)
        # ideal: all expected articles ranked first
        ideal_rel = 1.0 if rank <= len(expected_set) else 0.0
        idcg += (2**ideal_rel - 1) / np.log2(rank + 1)
    ndcg = dcg / idcg if idcg > 0 else 0.0

    # Hit@k
    hit = 1.0 if any(art in expected_set for art in top_k) else 0.0

    return {
        "precision@k": round(precision_at_k, 4),
        "mrr@k": round(mrr, 4),
        "ndcg@k": round(ndcg, 4),
        "hit@k": round(hit, 4),
    }


def normalize_article_number(art: str) -> str:
    """Normalize article numbers for fuzzy matching (strip leading zeros, etc.)."""
    return re.sub(r"\s+", "", str(art)).lstrip("0") or "0"


print("Context precision / relevance functions defined.")

Context precision / relevance functions defined.


## Section 4: Define Context Recall Metric

**Context Recall** measures whether the retriever found *all* the necessary documents:

$$\text{Recall@k} = \frac{|\text{relevant articles} \cap \text{retrieved articles}|}{|\text{relevant articles}|}$$

In [24]:
# =============================================================================
# Section 4: Context Recall Metric
# =============================================================================

def compute_context_recall(
    retrieved_articles: List[str],
    expected_articles: List[str],
    k: int = 5,
) -> float:
    """Compute Recall@k -- fraction of expected articles found in top-k results.

    Args:
        retrieved_articles: List of article numbers retrieved (ordered by rank).
        expected_articles: List of expected (ground-truth) article numbers.
        k: Cutoff rank.

    Returns:
        Recall@k score in [0, 1].
    """
    expected_set = set(expected_articles)
    if not expected_set:
        return 0.0

    top_k = set(retrieved_articles[:k])
    relevant_found = len(expected_set & top_k)

    return round(relevant_found / len(expected_set), 4)


print("Context recall function defined.")

Context recall function defined.


## Section 5: Define Answer Relevance Metric

**Answer Relevance** quantifies how well the generated answer addresses the user's question, using the retrieved context. We use semantic similarity (cosine between question and answer embeddings) as a proxy.

In [25]:
# =============================================================================
# Section 5: Answer Relevance Metric
# =============================================================================

# Lazy-init embedder for semantic comparisons
_answer_rel_embedder = None


def _get_answer_rel_embedder():
    global _answer_rel_embedder
    if _answer_rel_embedder is None:
        _answer_rel_embedder = create_embedder()
    return _answer_rel_embedder


def compute_answer_relevance(
    question: str,
    generated_answer: str,
    retrieved_contexts: List[str],
) -> float:
    """Compute answer relevance via semantic similarity.

    Score = cosine_similarity(embed(question), embed(generated_answer))
    Clamped to [0, 1].

    Args:
        question: The user's question in Arabic.
        generated_answer: The LLM-generated answer.
        retrieved_contexts: Retrieved document texts (for future LLM-as-judge).

    Returns:
        Relevance score in [0, 1].
    """
    if not generated_answer or not question:
        return 0.0

    try:
        embedder = _get_answer_rel_embedder()
        q_emb = embedder.embed_single(question)
        a_emb = embedder.embed_single(generated_answer)

        # Cosine similarity
        similarity = np.dot(q_emb, a_emb) / (
            np.linalg.norm(q_emb) * np.linalg.norm(a_emb) + 1e-9
        )
        return round(max(0.0, min(1.0, float(similarity))), 4)
    except Exception as e:
        print(f"[WARN] Answer relevance computation failed: {e}")
        return 0.0


print("Answer relevance function defined.")

Answer relevance function defined.


## Section 6: Define Faithfulness (Hallucination) Metric

**Faithfulness** checks whether the generated claims are grounded in the retrieved documents.

We use a lightweight approach:
1. Split the answer into sentences (atomic claims)
2. For each claim, check if it can be found (as a substring or via high semantic similarity) in the retrieved context
3. Faithfulness = `|grounded claims| / |total claims|`

In [26]:
# =============================================================================
# Section 6: Faithfulness (Hallucination) Metric
# =============================================================================

ARABIC_SENTENCE_SPLIT = re.compile(r"[.؟!؛\n]+")


def _split_arabic_sentences(text: str) -> List[str]:
    """Split Arabic text into atomic claims (sentences)."""
    parts = ARABIC_SENTENCE_SPLIT.split(text)
    return [p.strip() for p in parts if len(p.strip()) > 10]


def _claim_grounded(
    claim: str,
    contexts: List[str],
    min_overlap: float = 0.3,
) -> bool:
    """Check if a claim is grounded in at least one retrieved context.

    Uses a hybrid approach:
    1. Substring / token overlap (fast)
    2. Semantic similarity via embedding (fallback)
    """
    claim_tokens = set(claim.split())

    for ctx in contexts:
        # Check substring
        if claim[:50] in ctx or ctx[:50] in claim:
            return True

        # Token overlap
        ctx_tokens = set(ctx.split())
        if claim_tokens and ctx_tokens:
            overlap = len(claim_tokens & ctx_tokens) / len(claim_tokens)
            if overlap >= min_overlap:
                return True

    return False


def compute_faithfulness(
    generated_answer: str,
    retrieved_contexts: List[str],
) -> float:
    """Compute faithfulness score.

    Args:
        generated_answer: The LLM-generated answer.
        retrieved_contexts: List of retrieved document texts.

    Returns:
        Faithfulness score in [0, 1].
    """
    if not generated_answer:
        return 0.0

    claims = _split_arabic_sentences(generated_answer)
    if not claims:
        return 0.0

    grounded = sum(1 for c in claims if _claim_grounded(c, retrieved_contexts))
    return round(grounded / len(claims), 4)


print("Faithfulness functions defined.")

Faithfulness functions defined.


## Section 7: Define Answer Correctness Metric

**Answer Correctness** measures how closely the generated answer matches the ground-truth expected answer. We use ROUGE-L and BERTScore F1 (if available).

$$\text{Correctness} = \alpha \cdot \text{ROUGE-L} + (1 - \alpha) \cdot \text{BERTScore F1}$$

In [27]:
# =============================================================================
# Section 7: Answer Correctness Metric
# =============================================================================

_rouge_scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False) if _HAS_ROUGE else None


def compute_answer_correctness(
    generated_answer: str,
    expected_answer: str,
    alpha: float = 0.4,
) -> float:
    """Compute answer correctness via ROUGE-L + BERTScore.

    Args:
        generated_answer: The LLM-generated answer.
        expected_answer: The ground-truth expected answer.
        alpha: Weight for ROUGE-L vs BERTScore (alpha * rouge + (1-alpha) * bert).

    Returns:
        Correctness score in [0, 1].
    """
    if not generated_answer or not expected_answer:
        return 0.0

    # --- ROUGE-L ---
    rouge_l = 0.0
    if _rouge_scorer is not None:
        scores = _rouge_scorer.score(expected_answer, generated_answer)
        rouge_l = scores["rougeL"].fmeasure

    # --- BERTScore F1 ---
    bert_f1 = 0.0
    if _HAS_BERT_SCORE:
        try:
            P, R, F1 = bert_score_fn(
                [generated_answer],
                [expected_answer],
                lang="ar",
                verbose=False,
            )
            bert_f1 = float(F1[0])
        except Exception as e:
            print(f"[WARN] BERTScore failed, using ROUGE only: {e}")
            bert_f1 = rouge_l  # fallback

    # Combined score
    if _HAS_BERT_SCORE and bert_f1 > 0:
        correctness = alpha * rouge_l + (1 - alpha) * bert_f1
    else:
        correctness = rouge_l

    return round(max(0.0, min(1.0, correctness)), 4)


print("Answer correctness function defined.")

Answer correctness function defined.


## Section 8: Compute All Metrics on the Dataset

Run retrieval + generation for each question across all three strategies (dense, bm25, rrf), then compute all five metrics.

In [28]:
# =============================================================================
# Section 8: Compute All Metrics on the Dataset
# =============================================================================

STRATEGIES = ["dense", "bm25", "rrf"]
TOP_K = 5

# Initialize LLM once
try:
    llm = create_llm()
    print(f"LLM ready: {llm.name}")
except Exception as e:
    print(f"WARNING: LLM not available: {e}")
    print("   -> Running retrieval-only benchmark (no generation metrics).")
    llm = None

# Re-init embedder for consistent session
embedder = create_embedder()
print(f"Embedder ready: {embedder.mode} | dim={embedder.dimension}")


def build_rag_prompt(query: str, sources: list) -> str:
    """Build a grounded legal prompt with retrieved articles."""
    articles_text = []
    for i, src in enumerate(sources, 1):
        excerpt = src.get("text", "")[:800]
        articles_text.append(
            f"[{i}] المادة {src.get('article_number', '?')} "
            f"من {src.get('law_name', '?')}:\n{excerpt}\n"
        )
    context = "\n".join(articles_text)
    return (
        f"النصوص القانونية المسترجعة:\n\n{context}\n\n"
        f"بناء على النصوص القانونية أعلاه فقط، أجب على السؤال التالي "
        f"بشكل دقيق مع ذكر أرقام المواد القانونية:\n\n{query}"
    )


# =============================================================================
# Main evaluation loop
# =============================================================================
results = []

for idx, row in per_question.iterrows():
    qid = row["id"]
    domain = row["domain"]
    difficulty = row["difficulty"]
    question = row["question_ar"]
    expected_answer = row["expected_answer"]
    expected_arts = [normalize_article_number(a) for a in row["expected_articles"]]

    for strategy_name in STRATEGIES:
        t0 = time.time()

        # --- Retrieval ---
        try:
            resp = search(
                query=question,
                domain=domain,
                top_k=TOP_K,
                strategy=strategy_name,
            )
            search_time_ms = resp.search_time_ms
            retrieved_arts = [
                normalize_article_number(r.article_number)
                for r in resp.results
            ]
            retrieved_texts = [r.text for r in resp.results]
        except Exception as e:
            print(f"[WARN] Retrieval failed for {qid}/{strategy_name}: {e}")
            search_time_ms = 0
            retrieved_arts = []
            retrieved_texts = ["ERROR"]

        # --- Context Metrics ---
        prec = compute_context_precision(retrieved_arts, expected_arts, k=TOP_K)
        recall = compute_context_recall(retrieved_arts, expected_arts, k=TOP_K)

        # --- Generation ---
        generated_answer = ""
        generation_time_ms = 0.0
        if llm is not None and retrieved_texts:
            try:
                sources = [
                    {"article_number": r.article_number,
                     "law_name": r.law_name, "text": r.text}
                    for r in resp.results[:3]
                ]
                prompt = build_rag_prompt(question, sources)
                gen = llm.generate(prompt, max_tokens=512, temperature=0.3)
                generated_answer = gen.response
                generation_time_ms = gen.generation_time_ms
            except Exception as e:
                print(f"[WARN] Generation failed for {qid}/{strategy_name}: {e}")
                generated_answer = "[GENERATION ERROR]"

        # --- Answer Quality Metrics ---
        relevance = compute_answer_relevance(
            question, generated_answer, retrieved_texts,
        )
        faithfulness = compute_faithfulness(
            generated_answer, retrieved_texts,
        )
        correctness = compute_answer_correctness(
            generated_answer, expected_answer,
        )

        total_time_ms = (time.time() - t0) * 1000

        results.append({
            "id": qid,
            "domain": domain,
            "difficulty": difficulty,
            "strategy": strategy_name,
            "question": question[:80],
            "search_time_ms": round(search_time_ms, 2),
            "generation_time_ms": round(generation_time_ms, 2),
            "total_time_ms": round(total_time_ms, 2),
            "precision@k": prec["precision@k"],
            "mrr@k": prec["mrr@k"],
            "ndcg@k": prec["ndcg@k"],
            "hit@k": prec["hit@k"],
            "recall@k": recall,
            "answer_relevance": relevance,
            "faithfulness": faithfulness,
            "answer_correctness": correctness,
            "retrieved_articles": retrieved_arts,
            "expected_articles": expected_arts,
            "generated_answer": generated_answer[:300],
            "expected_answer": expected_answer[:300],
        })

        print(
            f"[{qid}] {strategy_name:5s} | "
            f"P@{TOP_K}={prec['precision@k']:.2f} "
            f"R@{TOP_K}={recall:.2f} "
            f"MRR={prec['mrr@k']:.2f} "
            f"F={faithfulness:.2f} "
            f"C={correctness:.2f}"
        )

df_results = pd.DataFrame(results)
print(f"\nBenchmark complete: {len(df_results)} rows "
      f"({len(per_question)} questions x {len(STRATEGIES)} strategies)")

# Save raw results
df_results.to_csv(OUTPUT_DIR / "benchmark_raw_results.csv", index=False)
print(f"Raw results -> {OUTPUT_DIR / 'benchmark_raw_results.csv'}")
df_results.head(10)

LLM ready: OpenAI (deepseek-v4-flash)


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 44119.69it/s]


Embedder ready: local | dim=1024


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 14508.62it/s]
[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[CIV-001] dense | P@5=0.20 R@5=1.00 MRR=1.00 F=0.25 C=0.47
[CIV-001] bm25  | P@5=0.00 R@5=0.00 MRR=0.00 F=0.00 C=0.00


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 13214.48it/s]
[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


... (122 outputs left out)

[TRAF-004] dense | P@5=0.00 R@5=0.00 MRR=0.00 F=0.60 C=0.51
[TRAF-004] bm25  | P@5=0.00 R@5=0.00 MRR=0.00 F=0.00 C=0.00


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 14352.20it/s]
[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[TRAF-004] rrf   | P@5=0.00 R@5=0.00 MRR=0.00 F=0.50 C=0.39

Benchmark complete: 96 rows (32 questions x 3 strategies)
Raw results -> /home/regisx001/Analyse-des-lois-justice-Maroc/benchmark/output/benchmark_raw_results.csv


,id,domain,difficulty,strategy,question,search_time_ms,generation_time_ms,total_time_ms,precision@k,mrr@k,ndcg@k,hit@k,recall@k,answer_relevance,faithfulness,answer_correctness,retrieved_articles,expected_articles,generated_answer,expected_answer
0,CIV-001,civil,easy,dense,هل يمكن للمحكمة دعوة الأطراف إلى الوساطة لحل ا...,5613.71,6099.24,20562.93,0.2,1.0000,1.0000,1.0,1.0,0.8435,0.2500,0.4703,"[2, 34, 255, 190, 127]",[2],### 1. ملخص الإجابة\nنعم، يمكن للمحكمة دعوة ال...,نعم، يمكن للمحكمة أن تدعو الأطراف إلى حل النزا...
1,CIV-001,civil,easy,bm25,هل يمكن للمحكمة دعوة الأطراف إلى الوساطة لحل ا...,11.06,0.00,11.18,0.0,0.0000,0.0000,0.0,0.0,0.0000,0.0000,0.0000,[],[2],,نعم، يمكن للمحكمة أن تدعو الأطراف إلى حل النزا...
2,CIV-001,civil,easy,rrf,هل يمكن للمحكمة دعوة الأطراف إلى الوساطة لحل ا...,5982.68,5211.30,13354.15,0.2,0.2500,0.4307,1.0,1.0,0.8306,0.5000,0.4402,"[255, 256, 128, 2, 34]",[2],### 1. ملخص الإجابة\nنعم، يمكن للمحكمة دعوة ال...,نعم، يمكن للمحكمة أن تدعو الأطراف إلى حل النزا...
3,CIV-002,civil,medium,dense,ما هي قواعد الاختصاص القيمي للمحاكم الابتدائية...,5603.03,5562.56,11175.33,0.2,1.0000,1.0000,1.0,1.0,0.0000,0.0000,0.0000,"[28, 59, 71, 63, 49]",[28],,تحكم المحكمة الابتدائية ابتدائيا وانتهائيا حتى...
4,CIV-002,civil,medium,bm25,ما هي قواعد الاختصاص القيمي للمحاكم الابتدائية...,7.83,0.00,7.92,0.0,0.0000,0.0000,0.0,0.0,0.0000,0.0000,0.0000,[],[28],,تحكم المحكمة الابتدائية ابتدائيا وانتهائيا حتى...
5,CIV-002,civil,medium,rrf,ما هي قواعد الاختصاص القيمي للمحاكم الابتدائية...,6082.50,5883.84,14819.20,0.2,0.3333,0.5000,1.0,1.0,0.7331,0.6250,0.7259,"[328, 616, 28, 59, 643]",[28],### 1. ملخص الإجابة\nبناء على النصوص القانونية...,تحكم المحكمة الابتدائية ابتدائيا وانتهائيا حتى...
6,CIV-003,civil,hard,dense,في أي الحالات تكون النيابة العامة طرفا أصليا ف...,6197.36,5912.82,12120.23,0.2,1.0000,1.0000,1.0,1.0,0.0000,0.0000,0.0000,"[14, 10, 12, 91, 609]",[14],,تكون النيابة العامة طرفا أصليا في القضايا المت...
7,CIV-003,civil,hard,bm25,في أي الحالات تكون النيابة العامة طرفا أصليا ف...,7.02,0.00,7.08,0.0,0.0000,0.0000,0.0,0.0,0.0000,0.0000,0.0000,[],[14],,تكون النيابة العامة طرفا أصليا في القضايا المت...
8,CIV-003,civil,hard,rrf,في أي الحالات تكون النيابة العامة طرفا أصليا ف...,5818.05,6302.59,12121.41,0.2,0.5000,0.6309,1.0,1.0,0.0000,0.0000,0.0000,"[10, 14, 12, 91, 398]",[14],,تكون النيابة العامة طرفا أصليا في القضايا المت...
9,CIV-004,civil,easy,dense,ما هي المدة المحددة لتبليغ الحكم الغيابي وما أ...,5497.31,5898.97,14115.48,0.0,0.0000,0.0000,0.0,0.0,0.7731,0.2857,0.4032,"[195, 88, 103, 554, 100]","[10, 391]",### 1. ملخص الإجابة\nالنصوص القانونية المقدمة ...,يبلغ منطوق الحكم الصادر غيابيا إلى الطرف المتغ...


## Section 9: Visualize Metrics Comparison

Generate rich comparative figures for the benchmark report.

In [29]:
# =============================================================================
# Section 9: Visualize Metrics Comparison
# =============================================================================

# --- Color palette consistent across figures ---
STRATEGY_COLORS = {"dense": "#2196F3", "bm25": "#FF9800", "rrf": "#4CAF50"}
STRATEGY_LABELS = {"dense": "Dense (Vector)", "bm25": "BM25 (Keyword)", "rrf": "RRF (Hybrid)"}

METRIC_COLUMNS = [
    ("precision@k", "Precision@5"),
    ("recall@k", "Recall@5"),
    ("mrr@k", "MRR@5"),
    ("ndcg@k", "NDCG@5"),
    ("hit@k", "Hit@5"),
    ("answer_relevance", "Answer Relevance"),
    ("faithfulness", "Faithfulness"),
    ("answer_correctness", "Answer Correctness"),
]

# Compute means per strategy
agg = df_results.groupby("strategy")[[col for col, _ in METRIC_COLUMNS]].mean().round(4)
print("Average metrics per strategy:")
print(agg.to_string())


# =============================================================================
# Figure 1: Grouped Bar Chart -- All Metrics per Strategy
# =============================================================================
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for ax, (col, title) in zip(axes, METRIC_COLUMNS):
    means = []
    colors = []
    for s in STRATEGIES:
        means.append(agg.loc[s, col])
        colors.append(STRATEGY_COLORS[s])

    bars = ax.bar(STRATEGIES, means, color=colors, edgecolor="white", linewidth=1.2)
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Score")

    # Value labels on bars
    for bar, val in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
                f"{val:.3f}", ha="center", va="bottom", fontsize=9, fontweight="bold")

    ax.set_xticklabels([STRATEGY_LABELS.get(s, s) for s in STRATEGIES],
                       rotation=15, ha="right", fontsize=9)

fig.suptitle("RAG Benchmark -- All Metrics by Retrieval Strategy",
             fontsize=16, fontweight="bold", y=1.01)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "fig1_all_metrics_bar.png", dpi=150)
plt.show()
print("Figure 1 saved: fig1_all_metrics_bar.png")


# =============================================================================
# Figure 2: Box Plots -- Metric Distributions per Strategy
# =============================================================================
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for ax, (col, title) in zip(axes, METRIC_COLUMNS):
    plot_data = [df_results[df_results["strategy"] == s][col].values for s in STRATEGIES]
    bp = ax.boxplot(plot_data, labels=STRATEGIES, patch_artist=True, widths=0.5)

    for patch, s in zip(bp["boxes"], STRATEGIES):
        patch.set_facecolor(STRATEGY_COLORS[s])
        patch.set_alpha(0.6)

    # Add individual points with jitter
    for i, s in enumerate(STRATEGIES):
        y = plot_data[i]
        x = np.random.normal(i + 1, 0.04, size=len(y))
        ax.scatter(x, y, alpha=0.5, s=20, color=STRATEGY_COLORS[s], edgecolors="white", linewidth=0.3)

    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_ylim(-0.05, 1.15)
    ax.set_ylabel("Score")

fig.suptitle("RAG Benchmark -- Metric Distributions by Strategy",
             fontsize=16, fontweight="bold", y=1.01)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "fig2_metric_distributions.png", dpi=150)
plt.show()
print("Figure 2 saved: fig2_metric_distributions.png")

Average metrics per strategy:
          precision@k  recall@k   mrr@k  ndcg@k   hit@k  answer_relevance  faithfulness  answer_correctness
strategy                                                                                                   
bm25           0.0000    0.0000  0.0000  0.0000  0.0000            0.0000        0.0000              0.0000
dense          0.1562    0.5677  0.5651  0.5665  0.6562            0.4220        0.2213              0.2647
rrf            0.1562    0.5938  0.4583  0.4744  0.6875            0.4047        0.2185              0.2519
Figure 1 saved: fig1_all_metrics_bar.png
Figure 2 saved: fig2_metric_distributions.png


In [30]:
# =============================================================================
# Figure 3: Radar Chart -- Overall Strategy Comparison
# =============================================================================

radar_metrics = ["Precision@5", "Recall@5", "MRR@5", "Hit@5",
                 "Relevance", "Faithfulness", "Correctness"]
radar_cols = ["precision@k", "recall@k", "mrr@k", "hit@k",
              "answer_relevance", "faithfulness", "answer_correctness"]

N = len(radar_metrics)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # close the loop

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

for s in STRATEGIES:
    values = agg.loc[s, radar_cols].tolist()
    values += values[:1]
    ax.fill(angles, values, alpha=0.15, color=STRATEGY_COLORS[s])
    ax.plot(angles, values, "o-", linewidth=2, color=STRATEGY_COLORS[s],
            label=STRATEGY_LABELS[s], markersize=5)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_metrics, fontsize=10)
ax.set_ylim(0, 1.0)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(["0.2", "0.4", "0.6", "0.8", "1.0"], fontsize=8, color="grey")
ax.set_title("RAG Performance Radar -- Strategy Comparison",
             fontsize=14, fontweight="bold", pad=25)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=10)
ax.grid(True, alpha=0.3)

fig.savefig(OUTPUT_DIR / "fig3_radar_comparison.png", dpi=150)
plt.show()
print("Figure 3 saved: fig3_radar_comparison.png")


# =============================================================================
# Figure 4: Heatmap -- Per-Domain Performance (RRF only)
# =============================================================================

rrf_data = df_results[df_results["strategy"] == "rrf"]
domain_agg = rrf_data.groupby("domain")[radar_cols].mean().round(3)

# Sort by average across all metrics
domain_agg["avg"] = domain_agg.mean(axis=1)
domain_agg = domain_agg.sort_values("avg", ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(
    domain_agg[radar_cols].T,
    annot=True,
    fmt=".3f",
    cmap="YlGnBu",
    vmin=0, vmax=1,
    linewidths=0.8,
    linecolor="white",
    xticklabels=domain_agg.index,
    yticklabels=radar_metrics,
    cbar_kws={"label": "Score", "shrink": 0.8},
    ax=ax,
)
ax.set_title("Per-Domain Performance -- RRF Strategy",
             fontsize=14, fontweight="bold")
ax.set_xlabel("Legal Domain", fontsize=11)
ax.set_ylabel("Metric", fontsize=11)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "fig4_domain_heatmap.png", dpi=150)
plt.show()
print("Figure 4 saved: fig4_domain_heatmap.png")

Figure 3 saved: fig3_radar_comparison.png
Figure 4 saved: fig4_domain_heatmap.png


In [31]:
# =============================================================================
# Figure 5: Per-Difficulty Breakdown (RRF)
# =============================================================================

difficulty_order = ["easy", "medium", "hard"]
diff_agg = rrf_data.groupby("difficulty")[radar_cols].mean().round(3)
diff_agg = diff_agg.reindex(difficulty_order)

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(radar_metrics))
width = 0.25

for i, diff in enumerate(difficulty_order):
    if diff not in diff_agg.index:
        continue
    vals = diff_agg.loc[diff, radar_cols].values
    bars = ax.bar(x + i * width, vals, width,
                  label=diff.capitalize(),
                  color=["#4CAF50", "#FF9800", "#F44336"][i],
                  alpha=0.85, edgecolor="white")

ax.set_xticks(x + width)
ax.set_xticklabels(radar_metrics, rotation=30, ha="right")
ax.set_ylabel("Score")
ax.set_ylim(0, 1.1)
ax.set_title("Performance by Difficulty Level -- RRF Strategy",
             fontsize=14, fontweight="bold")
ax.legend(title="Difficulty", fontsize=10)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "fig5_difficulty_breakdown.png", dpi=150)
plt.show()
print("Figure 5 saved: fig5_difficulty_breakdown.png")


# =============================================================================
# Figure 6: Latency Comparison -- Search & Generation Time
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Search time
search_agg = df_results.groupby("strategy")["search_time_ms"].mean()
colors_search = [STRATEGY_COLORS[s] for s in search_agg.index]
axes[0].bar(search_agg.index, search_agg.values, color=colors_search, edgecolor="white")
axes[0].set_title("Average Search Time (ms)", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Milliseconds")
for bar, val in zip(axes[0].patches, search_agg.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                 f"{val:.0f}ms", ha="center", fontsize=10)

# Generation time
gen_agg = df_results[df_results["generation_time_ms"] > 0].groupby("strategy")["generation_time_ms"].mean()
colors_gen = [STRATEGY_COLORS[s] for s in gen_agg.index]
axes[1].bar(gen_agg.index, gen_agg.values, color=colors_gen, edgecolor="white")
axes[1].set_title("Average Generation Time (ms)", fontsize=13, fontweight="bold")
axes[1].set_ylabel("Milliseconds")
for bar, val in zip(axes[1].patches, gen_agg.values):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 10,
                 f"{val:.0f}ms", ha="center", fontsize=10)

fig.suptitle("Latency Benchmarks", fontsize=15, fontweight="bold")
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "fig6_latency_comparison.png", dpi=150)
plt.show()
print("Figure 6 saved: fig6_latency_comparison.png")

Figure 5 saved: fig5_difficulty_breakdown.png
Figure 6 saved: fig6_latency_comparison.png


## Section 10: Export Benchmark Results for Report

Export aggregated metrics and summary tables for the final report. All results are saved to CSV and PNG files.

In [32]:
# =============================================================================
# Section 10: Export Benchmark Results for Report
# =============================================================================

# --- Summary Table (mean per strategy) ---
summary = agg.copy()
summary.index.name = "strategy"
summary.to_csv(OUTPUT_DIR / "benchmark_summary.csv")
print("Strategy-Level Summary:")
print(summary.round(3).to_string())

# --- Per-Question Table (RRF only, for detailed report) ---
detail_cols = [
    "id", "domain", "difficulty", "question",
    "precision@k", "recall@k", "mrr@k",
    "answer_relevance", "faithfulness", "answer_correctness",
    "search_time_ms", "generation_time_ms",
]
rrf_detail = rrf_data[detail_cols].sort_values(["domain", "id"])
rrf_detail.to_csv(OUTPUT_DIR / "benchmark_detail_rrf.csv", index=False)

# --- Winner per metric ---
print("\nBest Strategy per Metric:")
for col, label in METRIC_COLUMNS:
    best = agg[col].idxmax()
    best_val = agg.loc[best, col]
    print(f"  {label:25s} -> {STRATEGY_LABELS[best]:20s} ({best_val:.4f})")

# --- Overall winner (average of all 8 metrics) ---
agg["overall"] = agg[[c for c, _ in METRIC_COLUMNS]].mean(axis=1)
overall_winner = agg["overall"].idxmax()
print(f"\nOverall Winner: {STRATEGY_LABELS[overall_winner]} "
      f"(avg={agg.loc[overall_winner, 'overall']:.4f})")

# --- Export metadata ---
figures_generated = [
    "fig1_all_metrics_bar.png",
    "fig2_metric_distributions.png",
    "fig3_radar_comparison.png",
    "fig4_domain_heatmap.png",
    "fig5_difficulty_breakdown.png",
    "fig6_latency_comparison.png",
]
print(f"\nAll outputs saved to: {OUTPUT_DIR}")
print(f"   CSV files: benchmark_raw_results.csv, benchmark_summary.csv, benchmark_detail_rrf.csv")
print(f"   Figures ({len(figures_generated)}): {', '.join(figures_generated)}")
print("\nBenchmark report ready.")

Strategy-Level Summary:
          precision@k  recall@k  mrr@k  ndcg@k  hit@k  answer_relevance  faithfulness  answer_correctness
strategy                                                                                                 
bm25            0.000     0.000  0.000   0.000  0.000             0.000         0.000               0.000
dense           0.156     0.568  0.565   0.566  0.656             0.422         0.221               0.265
rrf             0.156     0.594  0.458   0.474  0.688             0.405         0.218               0.252

Best Strategy per Metric:
  Precision@5               -> Dense (Vector)       (0.1562)
  Recall@5                  -> RRF (Hybrid)         (0.5938)
  MRR@5                     -> Dense (Vector)       (0.5651)
  NDCG@5                    -> Dense (Vector)       (0.5665)
  Hit@5                     -> RRF (Hybrid)         (0.6875)
  Answer Relevance          -> Dense (Vector)       (0.4220)
  Faithfulness              -> Dense (Vector)       (